# Step 1: Data Understanding & Profiling
## Real-Time Ride Demand Prediction with Continual Learning
**Dataset**: NYC TLC High Volume For-Hire Vehicle (HVFHV) Trip Records (January 2025) (~20.4 million rows)

### Objectives:
1. Efficiently inspect the Parquet file without converting to CSV or loading all 20M rows into memory.
2. Profile dataset dimensions, column schemas, and data types.
3. Check missing value distributions across all columns.
4. Verify timestamp boundaries and identify out-of-boundary records.
5. Identify provider market shares and count unique pickup zones.
6. Establish memory-conscious data selection for downstream modeling.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

# Ensure src modules can be imported
sys.path.append(str(Path.cwd().parent))
from src.config import RAW_PARQUET_FILE, NOTEBOOKS_PARQUET_FILE, DATA_START_BOUND, DATA_END_BOUND
from src.data_loader import inspect_raw_metadata, get_raw_parquet_path

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Configuration and modules imported successfully.")


### 1. Inspect Parquet Metadata (Row Groups & Schema)
We inspect the file metadata using `pyarrow.parquet.ParquetFile`. This reads only the footer metadata (~a few kilobytes) instead of loading 20.4 million rows.


In [ ]:
meta = inspect_raw_metadata()
print(f"File path: {meta['file_path']}")
print(f"Total Rows: {meta['num_rows']:,}")
print(f"Total Row Groups: {meta['num_row_groups']}")
print(f"Total Columns: {meta['num_columns']}")
print("\nColumns and Datatypes:")
for col, dtype in meta['schema'].items():
    print(f"  - {col:<26}: {dtype}")


### 2. Selective Column Loading for Memory Efficiency
Instead of loading all 25 columns (~4.5 GB in RAM), we selectively inspect a representative subset of operational and target columns:
- `hvfhs_license_num`: Provider license (HV0003=Uber, HV0005=Lyft)
- `request_datetime`: Timestamp of ride request
- `pickup_datetime`: Actual passenger pickup
- `PULocationID`: Taxi Zone of pickup
- `trip_miles`: Distance of trip
- `base_passenger_fare`: Fare charged


In [ ]:
target_cols = [
    'hvfhs_license_num', 'request_datetime', 'pickup_datetime', 
    'PULocationID', 'trip_miles', 'base_passenger_fare'
]

raw_path = get_raw_parquet_path()
print(f"Loading {target_cols} from {raw_path}...")
sample_df = pd.read_parquet(raw_path, columns=target_cols)

print(f"Loaded DataFrame Shape: {sample_df.shape}")
print(f"Memory Usage: {sample_df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
sample_df.head(10)


### 3. Missing Value Profiling
We verify missing values for our operational columns. Notice that `request_datetime` and `PULocationID` have **zero** missing values!


In [ ]:
null_counts = sample_df.isnull().sum()
null_pct = (null_counts / len(sample_df)) * 100
missing_df = pd.DataFrame({'Missing_Count': null_counts, 'Missing_Pct': null_pct})
print(missing_df)


### 4. Timestamp Boundary Verification
The raw TLC data often contains trip records spanning into adjacent months. We check the exact date bounds of `request_datetime`.


In [ ]:
min_req = sample_df['request_datetime'].min()
max_req = sample_df['request_datetime'].max()
print(f"Min request_datetime: {min_req}")
print(f"Max request_datetime: {max_req}")

outside_jan = sample_df[(sample_df['request_datetime'] < DATA_START_BOUND) | 
                        (sample_df['request_datetime'] >= DATA_END_BOUND)]
print(f"Records outside January 2025: {len(outside_jan):,} ({len(outside_jan)/len(sample_df)*100:.4f}%)")


### 5. Provider Market Share & Pickup Zones
Inspect the distribution of High-Volume For-Hire Service providers:
- `HV0003`: Uber
- `HV0005`: Lyft


In [ ]:
provider_map = {'HV0003': 'Uber', 'HV0005': 'Lyft', 'HV0002': 'Juno', 'HV0004': 'Via'}
provider_counts = sample_df['hvfhs_license_num'].value_counts()
provider_summary = pd.DataFrame({
    'Provider_Name': provider_counts.index.map(provider_map),
    'Trip_Count': provider_counts.values,
    'Market_Share_Pct': (provider_counts.values / len(sample_df)) * 100
}, index=provider_counts.index)
print(provider_summary)

unique_zones = sample_df['PULocationID'].nunique()
print(f"\nUnique Pickup Zones (PULocationID): {unique_zones}")


### 6. Key Takeaways for Demand Modeling
1. **Zero Nulls in Core Features**: `request_datetime` and `PULocationID` are 100% complete.
2. **Strict Boundary Filtering**: Exactly 1,169 records fall outside January 2025 and must be excluded.
3. **High-Efficiency Column Loading**: We only need `['request_datetime', 'PULocationID']` for 15-minute demand aggregation, requiring < 250 MB RAM for 20.4M rows.
4. **Spatial Granularity**: 262 unique zones across NYC.
